# 1.0 Setup notebook

These cells are required to setup the notebook

## Install required packages

Install the required Python packages to run crap in the notebook

In [ ]:
%pip install -r requirements.txt

## Get an access token for the ARM API

Grab an access token for the ARM API from Entra ID. The access token needs to be associated with a user, service principal, or managed identity with appropriate permissions to run the cells in this template. Contributor on the subscription should be sufficient.

In [ ]:
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv('.env', override=True)

# Get a token for Microsoft Graph with the logged in user
credential = DefaultAzureCredential()
scopes = ["https://management.azure.com/.default"]

user_token = credential.get_token(*scopes)

In [ ]:
# Setup logging in debug mode
import logging
import sys

def configure_logging(level="DEBUG"):
    """This function configures logging for code being run based on the specified level.
    Args:
        level (str): The logging level to use (e.g., "DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL").
    """
    try:
        # Convert the level string to uppercase so it matches what the logging library expects
        logging_level = getattr(logging, level.upper(), None)

        # Setup a logging format
        logging.basicConfig(
            level=logging_level,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[logging.StreamHandler(sys.stdout)]
        )
    except Exception as e:
        print(f"Failed to set up logging: {e}", file=sys.stderr)
        sys.exit(1) 

configure_logging("ERROR")

# 2.0 Review an existing Foundry resource configured with a managed virtual network

The cells in this section can be used to explore a Foundry account configured with managed virtual network. You can explore the managed vnet configuration and its outbound rules.

## Get the Foundry Account

This cell will use the ARM REST API to print out the Foundry account resource

In [ ]:
import requests
import os
import json

# load environment variables from .env file
load_dotenv('.env', override=True)

def get_foundry_account(subscription: str, resource_group_name: str, foundry_account_name: str, token: str):
    url = f"https://management.azure.com/subscriptions/{subscription}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{foundry_account_name}"
    api_version = "2025-10-01-preview"

    response = requests.get(
        url = url,
        params = {"api-version": api_version},
        headers = 
        {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
    )

    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to get foundry account: {response.status_code} - {response.text}")

foundry_account = get_foundry_account(
    subscription=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("FOUNDRY_ACCOUNT_RESOURCE_GROUP"),
    foundry_account_name=os.getenv("FOUNDRY_ACCOUNT_NAME"),
    token=user_token.token
)

# Parse the json return and make it pretty
print(json.dumps(foundry_account, indent=4))


## Get managed virtual network

As of 8/6/2026, the managed virtual network is automatically provisioned when the user specifies to use managed vnet. It is named default.

This cell will print out the properties of the managed vnet resource.

In [ ]:
import requests
import os
import json

# load environment variables from .env file
load_dotenv('.env', override=True)

def get_managed_vnet(subscription: str, resource_group_name: str, foundry_account_name: str, token: str):
    url = f"https://management.azure.com/subscriptions/{subscription}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{foundry_account_name}/managedNetworks/default"
    api_version = "2025-10-01-preview"

    response = requests.get(
        url = url,
        params = {"api-version": api_version},
        headers = 
        {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
    )

    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to get managed virtual network: {response.status_code} - {response.text}")

managed_virtual_network = get_managed_vnet(
    subscription=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("FOUNDRY_ACCOUNT_RESOURCE_GROUP"),
    foundry_account_name=os.getenv("FOUNDRY_ACCOUNT_NAME"),
    token=user_token.token
)

# Parse the json return and make it pretty
print(json.dumps(managed_virtual_network, indent=4))


## Get managed virtual network outbound rules

Managed outbound rules are children of the managed virtual network. Outbound rules can be FQDN-based rules, service tag-based rules, or private endpoint rules. Private Endpoint rules create a private endpoint for a customer resource in the Microsoft-managed virtual network.

In [ ]:
import requests
import os
import json

# load environment variables from .env file
load_dotenv('.env', override=True)

def get_managed_vnet_outbound_rules(subscription: str, resource_group_name: str, foundry_account_name: str, token: str):
    url = f"https://management.azure.com/subscriptions/{subscription}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{foundry_account_name}/managedNetworks/default/outboundRules"
    api_version = "2025-10-01-preview"

    response = requests.get(
        url = url,
        params = {"api-version": api_version},
        headers = 
        {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
    )

    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to get managed virtual network outbound rules: {response.status_code} - {response.text}")

managed_virtual_network_outbound_rules = get_managed_vnet_outbound_rules(
    subscription=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("FOUNDRY_ACCOUNT_RESOURCE_GROUP"),
    foundry_account_name=os.getenv("FOUNDRY_ACCOUNT_NAME"),
    token=user_token.token
)

# Parse the json return and make it pretty
print(json.dumps(managed_virtual_network_outbound_rules, indent=4))


# 3.0 Manage outbound rules

The cells in this section can be used to managed outbound rules

## 3.1 Create a new outbound rule

This cell can be used to create a new outbound rule in the managed virtual network.

In [ ]:
import requests
import os
import json
import time

# load environment variables from .env file
load_dotenv('.env', override=True)

def get_managed_vnet_outbound_rule(subscription: str, resource_group_name: str, foundry_account_name: str, rule_name: str,token: str):
    url = f"https://management.azure.com/subscriptions/{subscription}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{foundry_account_name}/managedNetworks/default/outboundRules/{rule_name}"
    api_version = "2025-10-01-preview"

    response = requests.get(
        url = url,
        params = {"api-version": api_version},
        headers = 
        {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
    )

    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to get managed virtual network outbound rules: {response.status_code} - {response.text}")

def create_private_endpoint_outbound_rule(subscription: str, resource_group_name: str, foundry_account_name: str, rule_name: str, sub_resource_id: str, sub_resource_target: str, fqdns: list, token: str):
    url = f"https://management.azure.com/subscriptions/{subscription}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{foundry_account_name}/managedNetworks/default/outboundRules/{rule_name}"
    api_version = "2025-10-01-preview"

    body = {
        "properties": {
            "type": "PrivateEndpointOutboundRule",
            "category": "UserDefined",
            "destination": {
                "serviceResourceId": sub_resource_id,
                "subResourceTarget": sub_resource_target
            },
            "fqdns": fqdns
        }
    }

    response = requests.put(
        url = url,
        params = {"api-version": api_version},
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        },
        json = body
    )

    if response.status_code == 200:
        return response.json()
    elif response.status_code == 202:
        update_response = get_managed_vnet_outbound_rule(subscription, resource_group_name, foundry_account_name, rule_name, token)
        status = update_response.get("properties", {}).get("status", "")
        while status != "Active":
            time.sleep(5)
            update_response = get_managed_vnet_outbound_rule(subscription, resource_group_name, foundry_account_name, rule_name, token)
            status = update_response.get("properties", {}).get("status", "")
        return update_response
    else:
        raise Exception(f"Failed to create managed private endpoint outbound rule: {response.status_code} - {response.text}")

RULE_NAME = "AllowApiManagement"
SUB_RESOURCE_ID = "\/subscriptions/XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX/resourceGroups/myresourcegroup/providers/Microsoft.ApiManagement/service/myapim"
SUB_RESOURCE_TYPE = "Gateway"
FQDNS = ["mycustomapimdomain.com"]

managed_virtual_network_outbound_rules = create_private_endpoint_outbound_rule(
    subscription=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("FOUNDRY_ACCOUNT_RESOURCE_GROUP"),
    foundry_account_name=os.getenv("FOUNDRY_ACCOUNT_NAME"),
    rule_name=RULE_NAME,
    sub_resource_id=SUB_RESOURCE_ID,
    sub_resource_target=SUB_RESOURCE_TYPE,
    fqdns=FQDNS,
    token=user_token.token
)

# Parse the json return and make it pretty
print(json.dumps(managed_virtual_network_outbound_rules, indent=4))
